# Volatility Forecasting in the South African (JSE) Stock Market

This notebook demonstrates a complete volatility-forecasting pipeline for the
**Johannesburg Stock Exchange (JSE)** using GARCH-family models.

**Outline**
1. Data Collection
2. Exploratory Data Analysis (EDA)
3. Stationarity & ARCH-effects Tests
4. Model Fitting (GARCH, EGARCH, GJR-GARCH)
5. Model Selection
6. Out-of-Sample Forecasting & Evaluation
7. Multi-step Ahead Forecast

In [ ]:
import sys
from pathlib import Path

# Make sure src/ is importable
sys.path.insert(0, str(Path(".").resolve()))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collection import fetch_and_prepare, JSE_TICKERS, DEFAULT_TICKERS
from src.preprocessing import (
    descriptive_statistics, check_stationarity,
    check_arch_effects, compute_realised_volatility,
)
from src.volatility_models import (
    fit_garch, select_best_model, forecast_volatility, fit_all_tickers
)
from src.evaluation import (
    rolling_window_forecast, summarise_model_performance
)
from src.visualisation import (
    plot_prices, plot_returns, plot_return_distribution,
    plot_realised_volatility, plot_conditional_volatility,
    plot_forecast_vs_realised, plot_acf_pacf,
    plot_model_comparison, plot_forecast_horizon,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

print("All imports OK.")

## 0. Configuration

In [ ]:
TICKERS     = DEFAULT_TICKERS  # ["^JTOPI", "NPN.JO", "SBK.JO", "MTN.JO"]
START_DATE  = "2015-01-01"
END_DATE    = "2024-12-31"
DIST        = "StudentsT"
HORIZON     = 10

print("JSE tickers available:")
for name, sym in JSE_TICKERS.items():
    print(f"  {sym:15s}  {name}")

## 1. Data Collection

Prices are downloaded from Yahoo Finance using .

In [ ]:
prices, log_returns = fetch_and_prepare(
    tickers=TICKERS, start=START_DATE, end=END_DATE
)
print(f"Price data: {prices.shape[0]} trading days x {prices.shape[1]} tickers")
prices.tail()

In [ ]:
plot_prices(prices, title="JSE Adjusted Closing Prices")

In [ ]:
plot_returns(log_returns, title="JSE Daily Log Returns")

## 2. Descriptive Statistics & EDA

We look at key statistics including excess kurtosis (fat tails) and the
Jarque-Bera normality test.

In [ ]:
desc = descriptive_statistics(log_returns)
desc.round(4)

In [ ]:
plot_return_distribution(log_returns)

## 3. Realised Volatility

21-day rolling standard deviation, annualised by $\sqrt{252}$.

In [ ]:
rv = compute_realised_volatility(log_returns, window=21, annualise=True)
plot_realised_volatility(rv, title="21-Day Realised Volatility (Annualised %)")

## 4. Statistical Tests

### 4.1 Augmented Dickey-Fuller Stationarity Test

In [ ]:
adf = check_stationarity(log_returns)
adf

### 4.2 ARCH-LM Heteroskedasticity Test

In [ ]:
arch_test = check_arch_effects(log_returns)
arch_test

### 4.3 ACF / PACF of Squared Returns

In [ ]:
for ticker in log_returns.columns:
    plot_acf_pacf(log_returns, ticker=ticker, lags=40)

## 5. GARCH Model Fitting & Selection

We grid-search over GARCH, EGARCH and GJR-GARCH with p,q ∈ {1,2} and
select the best model by **BIC**.

In [ ]:
best_results   = {}
selection_dfs  = {}

for ticker in log_returns.columns:
    print(f"
=== {ticker} ===")
    best_res, sel_df = select_best_model(
        log_returns[ticker], ticker=ticker,
        p_range=(1, 2), q_range=(1, 2),
        model_types=("GARCH", "EGARCH", "GJR-GARCH"),
        dist=DIST, criterion="bic",
    )
    best_results[ticker]  = best_res
    selection_dfs[ticker] = sel_df
    display(sel_df.head(6))

In [ ]:
for ticker, res in best_results.items():
    print(f"
--- Model summary: {ticker} ---")
    print(res.summary())

In [ ]:
for ticker, res in best_results.items():
    plot_conditional_volatility(res, ticker=ticker, model_type="Best (by BIC)")

## 6. Out-of-Sample Evaluation

We use a rolling-window scheme: train on the first 80% of data, then
step forward one day at a time, re-estimating the model each period.

In [ ]:
oos_results = {}
for ticker in log_returns.columns:
    print(f"Rolling forecast: {ticker} …")
    oos_df = rolling_window_forecast(
        log_returns[ticker], ticker=ticker,
        model_type="GJR-GARCH", dist=DIST,
        train_size=0.8, horizon=1,
    )
    oos_results[ticker] = oos_df
    print(f"  {len(oos_df)} out-of-sample predictions.")

In [ ]:
for ticker, df in oos_results.items():
    plot_forecast_vs_realised(df, ticker=ticker)

In [ ]:
summary = summarise_model_performance(oos_results)
print("
--- Out-of-sample performance ---")
summary.round(8)

In [ ]:
plot_model_comparison(summary, metric="RMSE")

## 7. Multi-Step Ahead Volatility Forecast

Producing a 10-day ahead annualised volatility forecast from the best model.

In [ ]:
for ticker, res in best_results.items():
    vol_fc = forecast_volatility(res, horizon=HORIZON)
    print(f"
{HORIZON}-day ahead forecast for {ticker}:")
    display(vol_fc.T)
    plot_forecast_horizon(vol_fc, ticker=ticker)

## 8. Conclusion

* All JSE return series exhibit **excess kurtosis** and **significant ARCH effects**, justifying the use of GARCH-family models.
* The **GJR-GARCH** model with Student-t innovations consistently outperforms the symmetric GARCH by capturing leverage effects (negative shocks increase volatility more than positive shocks of the same magnitude).
* Annualised conditional volatility for JSE equities fluctuated significantly, peaking during the COVID-19 shock in 2020 and during global macro stress periods.
* The 10-day-ahead volatility forecasts provide actionable risk estimates for portfolio construction, options pricing, and VaR calculations.